# Modul 11: Metriken, Kreuzvalidierung, Suche und Erklärbarkeit

    **Notebooktyp:** Übungen mit ausführlichen Lösungen  
    **Vorlesungen dieses Moduls:** Metriken und Suche, Merkmale erklären  
    **Erwarteter Schwierigkeitsgrad:** Fortgeschritten  
    **Orientierungszeit:** etwa 125 bis 175 Minuten

    ## Überblick

    Sie bewerten Modelle mit mehreren Metriken, wählen passende Kreuzvalidierungssplitter und führen kleine leakage-sichere Hyperparametersuchen durch. Danach vergleichen Sie Merkmalsauswahl, Dimensionsreduktion und einfache globale Modellinterpretationen.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_11A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_11B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - Passende Metriken für Klassifikation und Regression auswählen.
- Leakage-sichere Kreuzvalidierung mit passenden Splittern durchführen.
- Hyperparameter mit kleinen Suchräumen optimieren und Ergebnisse dokumentieren.
- Merkmale innerhalb von scikit-learn-Pipelines konstruieren und auswählen.
- Dimensionsreduktion und Merkmalsauswahl für unterschiedliche Datenformen vergleichen.
- Modelle mit Koeffizienten, Importances und Abhängigkeitsplots interpretieren.

    ## Bewertete Fähigkeiten

    - Balanced Accuracy, Precision, Recall, F1, ROC-AUC und Average Precision
- StratifiedKFold, GroupKFold, TimeSeriesSplit und cross_validate
- GridSearchCV und RandomizedSearchCV mit Pipelines
- SelectKBest, PCA und polynomiale Merkmale
- Koeffizienten, Feature Importances und Partial Dependence

## Arbeitsanweisungen

Dieses Lösungsnotebook entspricht dem Übungsnotebook Aufgabe für Aufgabe. Führen Sie es von oben nach unten aus und vergleichen Sie nicht nur Endwerte, sondern auch Vorgehen, Formprüfungen, Datenaufteilung und Interpretation. Die Kommentare erklären bewusst auch typische Fehlerquellen und methodische Entscheidungen.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# Gemeinsames Setup für dieses Notebook
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification, make_regression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.inspection import PartialDependenceDisplay
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    GroupKFold,
    KFold,
    RandomizedSearchCV,
    StratifiedKFold,
    TimeSeriesSplit,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.decomposition import PCA

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore", category=FutureWarning)

X_metrics, y_metrics = make_classification(
    n_samples=650,
    n_features=12,
    n_informative=6,
    n_redundant=2,
    weights=[0.88, 0.12],
    class_sep=1.0,
    flip_y=0.03,
    random_state=RANDOM_SEED,
)
feature_names_11 = [f"feature_{i:02d}" for i in range(X_metrics.shape[1])]
groups_11 = np.repeat(np.arange(130), 5)

X_reg_11, y_reg_11 = make_regression(
    n_samples=360,
    n_features=5,
    n_informative=4,
    noise=12.0,
    random_state=RANDOM_SEED,
)

print("Setup abgeschlossen. Zufallsstartwert:", RANDOM_SEED)


## Aufgabe 1: Metriken, ROC und Precision-Recall vergleichen

    Erstellen Sie einen stratifizierten Train/Test-Split für `X_metrics`, `y_metrics` und trainieren Sie eine skalierte logistische Regression.

1. Berechnen Sie Accuracy, Balanced Accuracy, Precision, Recall, F1, ROC-AUC und Average Precision.
2. Erstellen Sie die Konfusionsmatrix und einen `classification_report`.
3. Zeichnen Sie ROC- und Precision-Recall-Kurve als getrennte Abbildungen.
4. Wiederholen Sie Precision, Recall und F1 mit Schwellenwert 0.30.
5. Begründen Sie, welche Metriken bei der ungleichen Klassenverteilung besonders informativ sind.

> **Hinweis:** Schwellenwertabhängige und schwellenwertfreie Metriken beantworten unterschiedliche Fragen.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_metrics, y_metrics, test_size=0.25,
    random_state=RANDOM_SEED, stratify=y_metrics
)

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Metriken, ROC und Precision-Recall vergleichen
#
# Ziel dieser Codezelle:
# Erstellen Sie einen stratifizierten Train/Test-Split für Xmetrics, ymetrics und
# trainieren Sie eine skalierte logistische Regression. 1. Berechnen Sie Accuracy,
# Balanced Accuracy, Precision, Recall, F1, ROC-AUC und Av...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X_metrics,
    y_metrics,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=y_metrics,
)

metric_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)),
    ]
)
metric_model.fit(X_train, y_train)

positive_probabilities = metric_model.predict_proba(X_test)[:, 1]
default_predictions = (positive_probabilities >= 0.50).astype(int)
lower_threshold_predictions = (positive_probabilities >= 0.30).astype(int)

metric_summary = pd.DataFrame(
    [
        {
            "threshold": 0.50,
            "accuracy": accuracy_score(y_test, default_predictions),
            "balanced_accuracy": balanced_accuracy_score(y_test, default_predictions),
            "precision": precision_score(y_test, default_predictions, zero_division=0),
            "recall": recall_score(y_test, default_predictions, zero_division=0),
            "f1": f1_score(y_test, default_predictions, zero_division=0),
            "roc_auc": roc_auc_score(y_test, positive_probabilities),
            "average_precision": average_precision_score(y_test, positive_probabilities),
        },
        {
            "threshold": 0.30,
            "accuracy": accuracy_score(y_test, lower_threshold_predictions),
            "balanced_accuracy": balanced_accuracy_score(y_test, lower_threshold_predictions),
            "precision": precision_score(y_test, lower_threshold_predictions, zero_division=0),
            "recall": recall_score(y_test, lower_threshold_predictions, zero_division=0),
            "f1": f1_score(y_test, lower_threshold_predictions, zero_division=0),
            "roc_auc": roc_auc_score(y_test, positive_probabilities),
            "average_precision": average_precision_score(y_test, positive_probabilities),
        },
    ]
)

print(metric_summary.round(3).to_string(index=False))
print("\nKonfusionsmatrix bei 0.50:")
print(confusion_matrix(y_test, default_predictions))
print("\nKlassifikationsbericht:")
print(classification_report(y_test, default_predictions, zero_division=0))

false_positive_rate, true_positive_rate, _ = roc_curve(
    y_test,
    positive_probabilities,
)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(false_positive_rate, true_positive_rate, label=f"AUC={roc_auc_score(y_test, positive_probabilities):.3f}")
ax.plot([0, 1], [0, 1], linestyle="--", label="Zufall")
ax.set_title("ROC-Kurve")
ax.set_xlabel("False-Positive-Rate")
ax.set_ylabel("True-Positive-Rate")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

precision_values, recall_values, _ = precision_recall_curve(
    y_test,
    positive_probabilities,
)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(recall_values, precision_values, label=f"AP={average_precision_score(y_test, positive_probabilities):.3f}")
ax.axhline(y_test.mean(), linestyle="--", label="Positivanteil")
ax.set_title("Precision-Recall-Kurve")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Reflexion zu Aufgabe 1

Bei stark ungleichen Klassen kann Accuracy hoch sein, obwohl positive Fälle häufig übersehen werden. Balanced Accuracy gewichtet beide Klassen gleich. Precision und Recall beschreiben unterschiedliche Fehlerkosten. ROC-AUC misst Rangordnung über alle Schwellenwerte, während Average Precision und die Precision-Recall-Kurve bei seltenen positiven Fällen oft direkter zeigen, wie viele positive Vorhersagen tatsächlich korrekt sind.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 2: Kreuzvalidierungssplitter passend einsetzen

    Vergleichen Sie vier Splitter:

- `KFold` für eine allgemeine Regression,
- `StratifiedKFold` für die ungleiche Klassifikation,
- `GroupKFold` mit `groups_11`,
- `TimeSeriesSplit` für die nach Index geordneten Regressionsdaten.

Verwenden Sie `cross_validate` mit mindestens zwei Metriken und speichern Sie Mittelwert und Standardabweichung der Validierungsergebnisse. Prüfen Sie bei `GroupKFold` ausdrücklich, dass keine Gruppe gleichzeitig in Train und Validierung liegt.

> **Hinweis:** Vorverarbeitung gehört in die Pipeline, damit sie innerhalb jedes Trainingsfolds neu gelernt wird.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Kreuzvalidierungssplitter passend einsetzen
#
# Ziel dieser Codezelle:
# Vergleichen Sie vier Splitter: - KFold für eine allgemeine Regression, -
# StratifiedKFold für die ungleiche Klassifikation, - GroupKFold mit groups11, -
# TimeSeriesSplit für die nach Index geordneten Regressionsdaten. V...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

result_rows = []

# Klassifikationspipeline: Skalierung wird innerhalb jedes Folds gefittet.
classification_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)),
    ]
)

stratified_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_SEED,
)
stratified_scores = cross_validate(
    classification_pipeline,
    X_metrics,
    y_metrics,
    cv=stratified_cv,
    scoring={"balanced_accuracy": "balanced_accuracy", "roc_auc": "roc_auc"},
    n_jobs=1,
)
result_rows.append(
    {
        "splitter": "StratifiedKFold",
        "metric_1_mean": stratified_scores["test_balanced_accuracy"].mean(),
        "metric_1_std": stratified_scores["test_balanced_accuracy"].std(),
        "metric_2_mean": stratified_scores["test_roc_auc"].mean(),
        "metric_2_std": stratified_scores["test_roc_auc"].std(),
    }
)

group_cv = GroupKFold(n_splits=5)
group_scores = cross_validate(
    classification_pipeline,
    X_metrics,
    y_metrics,
    groups=groups_11,
    cv=group_cv,
    scoring={"balanced_accuracy": "balanced_accuracy", "roc_auc": "roc_auc"},
    n_jobs=1,
)
result_rows.append(
    {
        "splitter": "GroupKFold",
        "metric_1_mean": group_scores["test_balanced_accuracy"].mean(),
        "metric_1_std": group_scores["test_balanced_accuracy"].std(),
        "metric_2_mean": group_scores["test_roc_auc"].mean(),
        "metric_2_std": group_scores["test_roc_auc"].std(),
    }
)

# Eine explizite Fold-Prüfung kontrolliert die Gruppenbedingung.
for train_idx, validation_idx in group_cv.split(X_metrics, y_metrics, groups_11):
    train_groups = set(groups_11[train_idx])
    validation_groups = set(groups_11[validation_idx])
    assert train_groups.isdisjoint(validation_groups)

regression_pipeline = Pipeline(
    [("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))]
)

regular_kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_SEED,
)
kfold_scores = cross_validate(
    regression_pipeline,
    X_reg_11,
    y_reg_11,
    cv=regular_kfold,
    scoring={"mae": "neg_mean_absolute_error", "r2": "r2"},
    n_jobs=1,
)
result_rows.append(
    {
        "splitter": "KFold",
        "metric_1_mean": -kfold_scores["test_mae"].mean(),
        "metric_1_std": kfold_scores["test_mae"].std(),
        "metric_2_mean": kfold_scores["test_r2"].mean(),
        "metric_2_std": kfold_scores["test_r2"].std(),
    }
)

time_cv = TimeSeriesSplit(n_splits=5)
time_scores = cross_validate(
    regression_pipeline,
    X_reg_11,
    y_reg_11,
    cv=time_cv,
    scoring={"mae": "neg_mean_absolute_error", "r2": "r2"},
    n_jobs=1,
)
result_rows.append(
    {
        "splitter": "TimeSeriesSplit",
        "metric_1_mean": -time_scores["test_mae"].mean(),
        "metric_1_std": time_scores["test_mae"].std(),
        "metric_2_mean": time_scores["test_r2"].mean(),
        "metric_2_std": time_scores["test_r2"].std(),
    }
)

cv_summary = pd.DataFrame(result_rows)
print(cv_summary.round(3).to_string(index=False))

### Reflexion zu Aufgabe 2

Splitter sind nicht beliebig austauschbar. Stratifikation schützt Klassenanteile, GroupKFold hält abhängige Gruppen getrennt und TimeSeriesSplit bewahrt Zukunftsrichtung. Die Standardabweichung über Folds zeigt, wie empfindlich die Bewertung auf den konkreten Split reagiert. Unterschiedliche Splitter erzeugen unterschiedliche, fachlich begründete Schätzfragen.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 3: Grid Search und Randomized Search dokumentieren

    Optimieren Sie eine skalierte logistische Regression auf `X_metrics`, `y_metrics` mit `StratifiedKFold` und der Zielmetrik `average_precision`.

1. Grid Search über `C=[0.05, 0.2, 1, 5]` und `class_weight=[None, "balanced"]`.
2. Randomized Search über dieselben Werte plus `solver=["liblinear", "lbfgs"]`, mit höchstens sechs Kandidaten.
3. Speichern Sie Rang, Mittelwert, Standardabweichung und Parameter der besten Ergebnisse in einer Tabelle.
4. Bewerten Sie den besten Grid-Search-Schätzer auf einem vorher zurückgehaltenen Testsatz.

> **Hinweis:** Der Testsatz darf weder Hyperparameterwahl noch Schwellenwertentscheidung beeinflussen.

In [ ]:
X_search_train, X_search_test, y_search_train, y_search_test = train_test_split(
    X_metrics, y_metrics, test_size=0.20,
    random_state=RANDOM_SEED, stratify=y_metrics
)

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Grid Search und Randomized Search dokumentieren
#
# Ziel dieser Codezelle:
# Optimieren Sie eine skalierte logistische Regression auf Xmetrics, ymetrics mit
# StratifiedKFold und der Zielmetrik averageprecision. 1. Grid Search über C=[0.05,
# 0.2, 1, 5] und classweight=[None, "balanced"]. 2. Rando...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

X_search_train, X_search_test, y_search_train, y_search_test = train_test_split(
    X_metrics,
    y_metrics,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=y_metrics,
)

search_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                max_iter=1500,
                random_state=RANDOM_SEED,
            ),
        ),
    ]
)
search_cv = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=RANDOM_SEED,
)

grid = GridSearchCV(
    estimator=search_pipeline,
    param_grid={
        "model__C": [0.05, 0.2, 1.0, 5.0],
        "model__class_weight": [None, "balanced"],
    },
    scoring="average_precision",
    cv=search_cv,
    n_jobs=1,
    return_train_score=True,
)
grid.fit(X_search_train, y_search_train)

randomized = RandomizedSearchCV(
    estimator=search_pipeline,
    param_distributions={
        "model__C": [0.02, 0.05, 0.2, 1.0, 5.0, 20.0],
        "model__class_weight": [None, "balanced"],
        "model__solver": ["liblinear", "lbfgs"],
    },
    n_iter=6,
    scoring="average_precision",
    cv=search_cv,
    random_state=RANDOM_SEED,
    n_jobs=1,
    return_train_score=True,
)
randomized.fit(X_search_train, y_search_train)

def top_search_rows(search_object, search_name: str, top_n: int = 5) -> pd.DataFrame:
    results = pd.DataFrame(search_object.cv_results_)
    selected = results[
        [
            "rank_test_score",
            "mean_test_score",
            "std_test_score",
            "mean_train_score",
            "params",
        ]
    ].sort_values("rank_test_score").head(top_n).copy()
    selected.insert(0, "search", search_name)
    return selected

search_summary = pd.concat(
    [
        top_search_rows(grid, "GridSearch"),
        top_search_rows(randomized, "RandomizedSearch"),
    ],
    ignore_index=True,
)

best_test_probabilities = grid.best_estimator_.predict_proba(X_search_test)[:, 1]
held_out_average_precision = average_precision_score(
    y_search_test,
    best_test_probabilities,
)

print("Beste Grid-Parameter:", grid.best_params_)
print("CV Average Precision:", round(grid.best_score_, 3))
print("Zurückgehaltener Test AP:", round(held_out_average_precision, 3))
print("\nSuchergebnisse:")
print(search_summary.to_string(index=False))

### Reflexion zu Aufgabe 3

Die Suche verwendet nur den Such-Trainingssatz und seine internen Folds. Der äußere Testsatz bleibt bis zur finalen Bewertung unberührt. Ein sehr großer Abstand zwischen mittlerem Trainings- und Validierungsscore kann auf Überanpassung hindeuten. Randomized Search prüft nur eine Stichprobe der möglichen Kombinationen und ist besonders bei größeren Suchräumen nützlich.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 4: Merkmalsauswahl, PCA und polynomiale Merkmale vergleichen

    Vergleichen Sie drei Pipelinevarianten mit stratifizierter Kreuzvalidierung:

1. Skalierung + logistische Regression auf allen Merkmalen.
2. Skalierung + `SelectKBest(f_classif, k=6)` + logistische Regression.
3. Skalierung + `PCA(n_components=6)` + logistische Regression.

Erstellen Sie zusätzlich für `X_reg_11` eine Ridge-Pipeline mit `PolynomialFeatures(degree=2, include_bias=False)` und vergleichen Sie deren CV-MAE mit einer linearen Ridge-Pipeline. Dokumentieren Sie Ausgabedimensionen und Metriken.

> **Hinweis:** Vergleichen Sie nicht nur den Score, sondern auch Ausgabedimension und Interpretierbarkeit.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Merkmalsauswahl, PCA und polynomiale Merkmale vergleichen
#
# Ziel dieser Codezelle:
# Vergleichen Sie drei Pipelinevarianten mit stratifizierter Kreuzvalidierung: 1.
# Skalierung + logistische Regression auf allen Merkmalen. 2. Skalierung +
# SelectKBest(fclassif, k=6) + logistische Regression. 3. Skalieru...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

classification_variants = {
    "alle Merkmale": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)),
        ]
    ),
    "SelectKBest": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("select", SelectKBest(score_func=f_classif, k=6)),
            ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)),
        ]
    ),
    "PCA": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=6, random_state=RANDOM_SEED)),
            ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)),
        ]
    ),
}

feature_rows = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
for name, pipeline in classification_variants.items():
    scores = cross_validate(
        pipeline,
        X_metrics,
        y_metrics,
        cv=cv,
        scoring={"ap": "average_precision", "balanced": "balanced_accuracy"},
        n_jobs=1,
    )
    pipeline.fit(X_metrics, y_metrics)

    if "select" in pipeline.named_steps:
        output_dimension = int(pipeline.named_steps["select"].get_support().sum())
    elif "pca" in pipeline.named_steps:
        output_dimension = int(pipeline.named_steps["pca"].n_components_)
    else:
        output_dimension = X_metrics.shape[1]

    feature_rows.append(
        {
            "variant": name,
            "output_features": output_dimension,
            "mean_AP": scores["test_ap"].mean(),
            "mean_balanced_accuracy": scores["test_balanced"].mean(),
        }
    )

regression_variants = {
    "linear Ridge": Pipeline(
        [("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))]
    ),
    "quadratische Merkmale": Pipeline(
        [
            ("poly", PolynomialFeatures(degree=2, include_bias=False)),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0)),
        ]
    ),
}
regression_rows = []
regression_cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
for name, pipeline in regression_variants.items():
    scores = cross_validate(
        pipeline,
        X_reg_11,
        y_reg_11,
        cv=regression_cv,
        scoring="neg_mean_absolute_error",
        n_jobs=1,
    )
    pipeline.fit(X_reg_11, y_reg_11)
    if "poly" in pipeline.named_steps:
        output_dimension = pipeline.named_steps["poly"].n_output_features_
    else:
        output_dimension = X_reg_11.shape[1]
    regression_rows.append(
        {
            "variant": name,
            "output_features": output_dimension,
            "mean_MAE": -scores["test_score"].mean(),
        }
    )

print("Klassifikation:")
print(pd.DataFrame(feature_rows).round(3).to_string(index=False))
print("\nRegression:")
print(pd.DataFrame(regression_rows).round(3).to_string(index=False))

### Reflexion zu Aufgabe 4

SelectKBest behält originale Merkmale und erleichtert die spätere Benennung. PCA erzeugt neue lineare Kombinationen, die Varianz erhalten, aber fachlich weniger direkt interpretierbar sind. Polynomiale Merkmale erhöhen die Dimension schnell und können nichtlineare Beziehungen abbilden, erhöhen jedoch auch Rechenaufwand und Überanpassungsrisiko. Alle Auswahl- und Konstruktionsschritte müssen innerhalb der Pipeline liegen.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 5: Integrationsaufgabe: Koeffizienten, Importances und Partial Dependence

    Trainieren Sie auf einem festen Split:

- eine skalierte logistische Regression,
- einen begrenzten Random Forest.

1. Erstellen Sie Ranglisten aus absoluten LogReg-Koeffizienten und Forest-Importances.
2. Vergleichen Sie die Top-5-Merkmale beider Modelle.
3. Erstellen Sie einen Partial-Dependence-Plot für das wichtigste Forest-Merkmal.
4. Schreiben Sie eine kurze Erklärung, was jede Methode zeigt und welche Grenzen sie besitzt.

> **Hinweis:** Interpretieren Sie immer das konkrete Modell und seine Datenbasis, nicht vermeintliche Naturgesetze.

In [ ]:
X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X_metrics, y_metrics, test_size=0.25,
    random_state=RANDOM_SEED, stratify=y_metrics
)

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Integrationsaufgabe: Koeffizienten, Importances und Partial Dependence
#
# Ziel dieser Codezelle:
# Trainieren Sie auf einem festen Split: - eine skalierte logistische Regression, -
# einen begrenzten Random Forest. 1. Erstellen Sie Ranglisten aus absoluten LogReg-
# Koeffizienten und Forest-Importances. 2. Vergleichen S...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X_metrics,
    y_metrics,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=y_metrics,
)

explanation_logreg = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)),
    ]
)
explanation_forest = RandomForestClassifier(
    n_estimators=180,
    max_depth=7,
    min_samples_leaf=4,
    random_state=RANDOM_SEED,
    n_jobs=1,
)

explanation_logreg.fit(X_train_e, y_train_e)
explanation_forest.fit(X_train_e, y_train_e)

coefficients = explanation_logreg.named_steps["model"].coef_[0]
coefficient_table = (
    pd.DataFrame(
        {
            "feature": feature_names_11,
            "coefficient": coefficients,
            "absolute_value": np.abs(coefficients),
        }
    )
    .sort_values("absolute_value", ascending=False)
    .reset_index(drop=True)
)

importance_table = (
    pd.DataFrame(
        {
            "feature": feature_names_11,
            "importance": explanation_forest.feature_importances_,
        }
    )
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

top_forest_feature = importance_table.loc[0, "feature"]
top_forest_index = feature_names_11.index(top_forest_feature)

print("Top LogReg-Koeffizienten:")
print(coefficient_table.head(5).round(3).to_string(index=False))
print("\nTop Forest-Importances:")
print(importance_table.head(5).round(3).to_string(index=False))
print("\nTest Balanced Accuracy:")
print(
    "LogReg =",
    round(balanced_accuracy_score(y_test_e, explanation_logreg.predict(X_test_e)), 3),
    "Forest =",
    round(balanced_accuracy_score(y_test_e, explanation_forest.predict(X_test_e)), 3),
)

# Partial Dependence mittelt Modellvorhersagen über die Verteilung der
# übrigen Merkmale. Sie beschreibt das Modell, nicht automatisch die Realität.
PartialDependenceDisplay.from_estimator(
    explanation_forest,
    X_train_e,
    features=[top_forest_index],
    feature_names=feature_names_11,
)
plt.tight_layout()
plt.show()

### Reflexion zu Aufgabe 5

LogReg-Koeffizienten zeigen Richtung und Stärke im standardisierten linearen Modell, unter Kontrolle der übrigen Modellterme. Forest-Importances messen, wie stark Merkmale zu Baumaufteilungen beigetragen haben, und können bei korrelierten oder hochvariablen Merkmalen verzerrt sein. Partial Dependence zeigt die durchschnittliche Modellreaktion, kann aber unrealistische Merkmalskombinationen erzeugen und beweist keine Kausalität.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Abschluss und Selbstkontrolle

Prüfen Sie nach dem Durcharbeiten, ob Sie jede Lösung ohne bloßes Kopieren erklären könnten. Achten Sie besonders auf die Stellen, an denen Datenleckage, unpassende Formen, falsche Metriken oder unkontrollierte Zufälligkeit zu scheinbar guten, aber methodisch falschen Ergebnissen führen könnten.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.